# Mesh and Skeleton from CloudVolume

CloudVolume: https://github.com/seung-lab/cloud-volume

In [ ]:
# !pip install cloud-volume

In [ ]:
import numpy as np
from cloudvolume import CloudVolume
from tqdm import tqdm

import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
import vtk

In [ ]:
# Dauer 364

vol = CloudVolume("gs://your_cloud/seg_oct22_nocb", mip=(4,4,50),
                  parallel=1, progress=True,
                  secrets=secrets)

print(vol.shape)

In [ ]:
# download multiple mesh (Dauer 364)

seg_id1_all = [73,74,75,76,48,40,36,37,35,49,17,18,16,15,26,25,23,24,20,19,
               22,21,83,29,27,30,28,63,34,54,39,51,43,50,32,33,45,52,31,44,
               70,69,81,80,85,86,93,87,84,82,92,91,96,97,95,94,89,90,88,98,
               42,47,41,46,38,53,220,218,58,61,187,186,225,219,71,57,78,79,
               198,181,180,194,193,207,208,183,182,197,196,209,205,201,199,
               189,188,217,213,60,59,68,67,62,77,66,65,56,55,64,72,142,143,
               147,146,149,150,144,156,155,158,157,145,152,151,154,153,148,
               159,160,161,173,172,166,167,175,174,164,163,162,165,171,170,
               177,176,169,168,235,234,232,233,240,239,237,236,248,247,229,
               246,238,245,228,227,230,226,242,241,243,185,184,216,206,204,
               192,178,179,191,190,221,222]  # 178

seg_id1_noci = [43,50, 81,80, 85,86, 93,87, 68,67]  # 10


# get mesh
dauer_mesh_all = vol.mesh.get(seg_id1_all, fuse=False)
dauer_mesh_noci = vol.mesh.get(seg_id1_noci, fuse=False)


In [ ]:
dauer_mesh_all[43].faces

In [ ]:
dauer_mesh_all[43].faces.shape

In [ ]:
dauer_mesh_all[43].vertices

In [ ]:
dauer_mesh_all[43].vertices.shape

In [ ]:
# =========================================================================================================
# =========================================================================================================

def triangle_area_3d(p1, p2, p3):
    """
    Calculates the area of a triangle in 3D space given three points.

    Args:
        p1, p2, p3: Coordinates of the three vertices as lists or numpy arrays
                    (e.g., [x, y, z]).

    Returns:
        The area of the triangle (float).
    """
    # Convert points to numpy arrays for vector operations
    v1 = np.array(p1)
    v2 = np.array(p2)
    v3 = np.array(p3)

    # Create two vectors sharing a common vertex (e.g., v1)
    vector_a = v2 - v1
    vector_b = v3 - v1

    # Calculate the cross product of the two vectors
    cross_product = np.cross(vector_a, vector_b)

    # The magnitude of the cross product is twice the area of the triangle
    # Calculate the magnitude and divide by 2
    area = np.linalg.norm(cross_product) / 2.0

    return area

# =========================================================================================================
# =========================================================================================================

In [ ]:
# get total surface area of each neurons in the list (Dauer)


dauer_surf_all = []


for ID in tqdm(seg_id1_all):
    
    FACE = dauer_mesh_all[ID].faces
    VER = dauer_mesh_all[ID].vertices
    
    AREA_sum = 0
    for ii in range(FACE.shape[0]):
        FF = FACE[ii,:]
        P1 = VER[FF[0],:]; P2 = VER[FF[1],:]; P3 = VER[FF[2],:]
        
        AREA = triangle_area_3d(P1, P2, P3)
        AREA_sum += AREA
    
    dauer_surf_all.append(AREA_sum)

print(len(dauer_surf_all))


In [ ]:
# get skeleton

dauer_skel_all = vol.skeleton.get(seg_id1_all)
dauer_skel_noci = vol.skeleton.get(seg_id1_noci)


In [ ]:
# get total skeleton length of each neurons in the list (Dauer)


dauer_len_all = np.zeros(len(dauer_skel_all))


idx_temp = np.array(seg_id1_all)

for ii in tqdm(range(len(dauer_skel_all))):
    PICK = dauer_skel_all[ii]
    ID = PICK.id
    ID_idx = np.where(idx_temp==ID)[0]
    if ID_idx.size!=1:
        print("check ", ID)
    else:
        dauer_len_all[ID_idx[0]] = PICK.cable_length()


print(np.where(dauer_len_all==0))  # should be an empty array


In [ ]:
# get total normalized 'surface area / unit length' of each neurons in the list (Dauer)


dauer_nSurf_all = np.copy(np.array(dauer_surf_all) / dauer_len_all)

print(dauer_nSurf_all.shape)


---

In [ ]:
# Dataset8 (Adult)

vol2 = CloudVolume("gs://your_cloud/dataset8/seg_v2", mip=4,
                   parallel=1, progress=True,
                   secrets=secrets)

print(vol2.shape)

In [ ]:
# download multiple mesh (Adult)

seg_id2_all = [133,169,80,184,70,112,151,42,84,51,116,148,76,187,4,66,10,
               170,127,45,25,111,131,123,211,179,14,150,78,140,27,193,60,
               113,53,172,182,6,109,104,114,145,108,26,220,217,91,30,218,
               216,205,68,110,37,221,219,203,156,97,102,128,71,135,134,87,
               15,98,154,190,130,41,34,202,121,164,198,176,67,64,171,103,
               20,5,13,19,197,105,39,136,96,161,200,201,16,115,118,94,208,
               119,124,9,12,185,204,117,32,90,206,166,189,99,55,100,81,79,
               7,29,40,199,22,92,49,1,59,74,142,160,2,147,73,75,159,191,139,
               83,120,65,183,122,157,162,23,43,89,17,11,50,77,86,158,44,93,
               129,174,46,54,106,155,192,35,196,153,18,58,168,62,95,173,163,
               24,167,38,57,132,107,33,72,181]  # 178

seg_id2_noci = [60,113, 108,26, 220,217, 91,30, 124,9]  # 10



In [ ]:
# get mesh

adult_mesh_all = vol2.mesh.get(seg_id2_all, fuse=False)  # check 'vol2' and 'seg_id2'
adult_mesh_noci = vol2.mesh.get(seg_id2_noci, fuse=False)  # check 'vol2' and 'seg_id2'


In [ ]:
adult_mesh_all[60].faces

In [ ]:
adult_mesh_all[60].faces.shape

In [ ]:
adult_mesh_all[60].vertices

In [ ]:
adult_mesh_all[60].vertices.shape

In [ ]:
# get total surface area of each neurons in the list (Adult)


adult_surf_all = []


for ID in tqdm(seg_id2_all):
    
    FACE = adult_mesh_all[ID].faces
    VER = adult_mesh_all[ID].vertices
    
    AREA_sum = 0
    for ii in range(FACE.shape[0]):
        FF = FACE[ii,:]
        P1 = VER[FF[0],:]; P2 = VER[FF[1],:]; P3 = VER[FF[2],:]
        
        AREA = triangle_area_3d(P1, P2, P3)
        AREA_sum += AREA
    
    adult_surf_all.append(AREA_sum)

print(len(adult_surf_all))


In [ ]:
# get skeleton

adult_skel_all = vol2.skeleton.get(seg_id2_all)  # check 'vol2' and 'seg_id2'
adult_skel_noci = vol2.skeleton.get(seg_id2_noci)  # check 'vol2' and 'seg_id2'


In [ ]:
# get total skeleton length of each neurons in the list (Adult)


adult_len_all = np.zeros(len(adult_skel_all))


idx_temp = np.array(seg_id2_all)

for ii in tqdm(range(len(adult_skel_all))):
    PICK = adult_skel_all[ii]
    ID = PICK.id
    ID_idx = np.where(idx_temp==ID)[0]
    if ID_idx.size!=1:
        print("check ", ID)
    else:
        adult_len_all[ID_idx[0]] = PICK.cable_length()


print(np.where(adult_len_all==0))  # should be an empty array


In [ ]:
# get total normalized 'surface area / unit length' of each neurons in the list (Adult)


adult_nSurf_all = np.copy(np.array(adult_surf_all) / adult_len_all)

print(adult_nSurf_all.shape)


## Neuron-wise area/length comparison

In [ ]:
# common type

import pandas as pd


com_fname = r'your_directory\230724_neurontype_new.csv'
com_id_df = pd.read_csv(com_fname)
com_id = np.array(com_id_df[['Neuron','Common_id']])


In [ ]:
# statistical test

from scipy.stats import wilcoxon


print("Area/length :", wilcoxon(adult_nSurf_all, dauer_nSurf_all)[1])


In [ ]:
# plot - all neurons


N_name = list(com_id[:78,0])
N_name = N_name + list(com_id[79:167,0])
N_name = N_name + list(com_id[168:180,0])
print(len(N_name))


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(24,4))

temp_x = np.arange(1,179,1)  # 178 neurons
ax1.scatter(temp_x, adult_nSurf_all, marker='o', c="sandybrown", alpha=0.7, s=20, label="Adult")
ax1.scatter(temp_x, dauer_nSurf_all, marker='s', c="tomato", alpha=0.7, s=20, label="Dauer")

ax1.set_xlabel("Neurons", fontsize=18, fontname='Arial')
ax1.set_ylabel("Area / length", fontsize=18, fontname='Arial')

ax1.set_xticks(temp_x)
ax1.set_xticklabels(N_name, fontsize=10, fontname='Arial', rotation=90)

ax1.set_xlim(0,179)
ax1.set_ylim(430,2110)

ax1.legend(bbox_to_anchor=(1.003, 1.010), loc='upper left', fontsize=16)
plt.tight_layout()

plt.show()


In [ ]:
# plot - nociception circuit


N_noci = ["ASHL","ASHR", "AVAL","AVAR", "AVBL","AVBR", "AVDL","AVDR", "PVCL","PVCR"]
noci_sel = np.array([32,33, 42,43, 44,45, 46,47, 100,101])


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(8,4))

temp_x = np.arange(1,11,1)  # 10 neurons
ax1.scatter(temp_x, adult_nSurf_all[noci_sel], marker='o', c="sandybrown", alpha=0.7, s=50, label="Adult")
ax1.scatter(temp_x, dauer_nSurf_all[noci_sel], marker='s', c="tomato", alpha=0.7, s=30, label="Dauer")

ax1.set_xlabel("Neurons", fontsize=18, fontname='Arial')
ax1.set_ylabel("Area / length", fontsize=18, fontname='Arial')

ax1.set_xticks(temp_x)
ax1.set_xticklabels(N_noci, fontsize=14, fontname='Arial', rotation=90)

ax1.set_xlim(0.5,10.5)
ax1.set_ylim(420,1460)

ax1.legend(bbox_to_anchor=(1.00, 1.00), loc='upper right', fontsize=14)
plt.tight_layout()

plt.show()


In [ ]:
print("Adult mean nSurf :", np.round(np.mean(adult_nSurf_all), 4))
print("Dauer mean nSurf :", np.round(np.mean(dauer_nSurf_all), 4))

From Pechuk et al. methods : "To maintain a relationship between Rm and Rin, we chose to fix the surface area of the neurons and set it to be 15 x 10^-6 cm2 as was observed in Rakowski et al., Roehrig, and Wicks et al."

In [ ]:
pechuk_surf_adult = 15 * 10**(-6)

pechuk_surf_dauer = pechuk_surf_adult * ( np.mean(dauer_nSurf_all) / np.mean(adult_nSurf_all) )
print("Dauer adjusted :", pechuk_surf_dauer)

---